# The Damiaatjes of Haarlem — rhythmic analysis of an evening ringing

Every evening 21:00–21:30 the two *Damiaatjes* bells of the Grote Kerk
(St. Bavo) in Haarlem ring, commemorating the Siege of Damietta (1218) and,
originally, the closing of the city gates. This notebook analyzes one such
evening (2026-07-17), recorded in first-order ambisonics (Zoom H3-VR, AmbiX
B-format, 48 kHz/24-bit), using the
[ambiscape](https://github.com/fourMs/ambiscape) toolbox.

The headline findings:

- two bells (strike notes ≈ B♭5 and D6) repeat a fixed **3-strike pattern
  every 3.1669 s** — long–short–long (1.37 / 0.42 / 1.38 s), B♭ … D–B♭;
- the two bells are **phase-locked** (relative phase R = 0.998, circular
  σ = 31 ms): one mechanically coordinated system;
- all free variation lives in bell A's **backswing strike** (115 ms
  cycle-to-cycle jitter, 9% misses) — repetition with variation, localized;
- the recorder clock was 665 s slow, calibrated against the known 21:30:00
  end of ringing (`calibration.json` → `clock_offset_s`).

**Requirements**: `pip install ambiscape` (≥ the commit adding the `rhythm`
module) and the session WAV `260717_001.WAV` in this folder (not in the git
repo — see README).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import ambiscape as asc
from ambiscape import rhythm, features, analysis

SESSION = Path(".")
sess = asc.open_session(SESSION)          # applies clock_offset_s
take = sess.takes[0]
print(f"{take.path.name}: start {sess.clock(take.start)} (corrected), "
      f"{take.duration/60:.1f} min, {take.channels}ch @{take.samplerate}")

260717_001.WAV: start 17 Jul 21:08:54 (corrected), 24.5 min, 4ch @48000


## 1. Session features

`ambiscape analyze` extracts per-second features in one streaming pass and
caches them (`analysis/features/*.npz`). We reuse the cache here; the
session-level descriptors summarize the whole 24.5 minutes.

In [2]:
paths = features.extract_session(sess, SESSION / "analysis" / "features")
F = features.load_features(paths)
summary = analysis.summarize(F)
{k: v for k, v in summary.items()}

{'duration_min': 24.5,
 'leq_dbfs': -38.6,
 'laeq_dbfs': -40.1,
 'leq_minus_laeq_db': 1.5,
 'L10': -40.9,
 'L50': -43.7,
 'L90': -45.4,
 'dynamics_L10_L90': 4.5,
 'events_per_min': 1.3,
 'event_median_dur_s': 0.25,
 'centroid_median_hz': 532,
 'flatness_median': 0.007,
 'diffuseness_median': 0.63,
 'diffuseness_iqr': 0.12,
 'azimuth_mean_deg': 52.0,
 'azimuth_R': 0.95,
 'azimuth_fg_deg': 55.0,
 'elevation_fg_median_deg': -3.0,
 'n_events': 31}

## 2. Finding the bell partials

The cached features include a full-resolution mean power spectrum per minute
(`minspec`). Contrasting bell-active minutes against the quiet tail exposes
the bells' narrowband partials without another audio pass. The ringing stops
at file-time 1266 s (= 21:30:00 — that is how the clock offset was
calibrated).

In [3]:
T_STOP = 1266.0
nmin = F["minspec"].shape[0]
active = np.zeros(nmin, bool); active[:20] = True
quiet = np.zeros(nmin, bool);  quiet[22:] = True
pfreq, rise = rhythm.detect_partials(F, active, quiet)
for f, r in zip(pfreq, rise):
    print(f"  {f:7.1f} Hz  +{r:5.1f} dB above quiet ambience")

    474.6 Hz  +  9.9 dB above quiet ambience
    597.7 Hz  + 13.3 dB above quiet ambience
    919.9 Hz  + 11.5 dB above quiet ambience
   1054.7 Hz  + 10.7 dB above quiet ambience
   1130.9 Hz  + 20.3 dB above quiet ambience
   1189.5 Hz  +  6.4 dB above quiet ambience
   1365.2 Hz  + 20.2 dB above quiet ambience
   1582.0 Hz  + 11.0 dB above quiet ambience
   1816.4 Hz  +  7.1 dB above quiet ambience
   1910.2 Hz  + 23.2 dB above quiet ambience
   2302.7 Hz  + 21.2 dB above quiet ambience
   2343.8 Hz  + 12.8 dB above quiet ambience
   2437.5 Hz  + 10.4 dB above quiet ambience
   2466.8 Hz  +  9.6 dB above quiet ambience
   2625.0 Hz  +  7.4 dB above quiet ambience
   2871.1 Hz  + 27.6 dB above quiet ambience
   3029.3 Hz  + 12.3 dB above quiet ambience
   3070.3 Hz  +  9.7 dB above quiet ambience
   3369.1 Hz  + 17.7 dB above quiet ambience
   3427.7 Hz  + 22.7 dB above quiet ambience
   3650.4 Hz  +  9.4 dB above quiet ambience
   3861.3 Hz  + 12.3 dB above quiet ambience
   3966.8 

## 3. One high-resolution pass, then sources

`rhythm.partial_pass` streams the file once more, storing per 20 ms frame the
power envelope and pseudo-intensity at each partial. Partials whose
log-envelope derivatives rise together belong to the same bell
(`rhythm.cluster_partials`).

In [4]:
P = rhythm.partial_pass(take, pfreq)
t = P["t"]; dt = float(np.median(np.diff(t)))
groups = rhythm.cluster_partials(P["env"], mask=t < T_STOP)[:2]
for gi, cols in enumerate(groups):
    print(f"bell {'AB'[gi]}: partials",
          [round(float(pfreq[c])) for c in cols])

bell A: partials [1131, 1910, 2344, 2438, 2467, 2871, 3369, 3861, 3967, 3996]
bell B: partials [1365, 2303, 3029, 3428]


## 4. Strikes and periodicity

Onset picking is ACF-informed (the separation guard comes from the onset
function's own autocorrelation), and the cycle period is refined with a
Rayleigh point-process periodogram near the ACF fundamental — both guards
against subharmonics. Both bells land on the **same** period, 3.1669 s.

In [5]:
strikes, periods = {}, {}
for gi, cols in enumerate(groups):
    name = "AB"[gi]
    odf = rhythm.source_odf(P["env"], cols)
    cycle, min_gap = rhythm.acf_structure(odf, dt, t_mask=t < T_STOP)
    s = rhythm.pick_strikes(odf, t, min_sep=0.8 * min_gap, t_max=T_STOP)
    Pbest = rhythm.best_period(s, lo=0.9 * cycle, hi=1.1 * cycle)
    strikes[name], periods[name] = s, Pbest
    print(f"bell {name}: {len(s)} strikes, ACF cycle {cycle:.3f} s, "
          f"refined period {Pbest:.4f} s")

bell A: 895 strikes, ACF cycle 3.140 s, refined period 3.1669 s


bell B: 771 strikes, ACF cycle 3.200 s, refined period 3.1670 s


### Phase folds

Strike times folded at the common period: each bell draws stable lines whose
slow common bending is the shared clock wander. Bell A has two lines (swing +
backswing), bell B one genuine line (its second line, coincident with A's
primary, is spectral cross-talk — verified below).

In [6]:
Pc = periods["A"]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for a, name, c in zip(ax, "AB", ["#2a78d6", "#d66a2a"]):
    tk = strikes[name]
    a.plot(tk, (tk / Pc) % 1.0, ".", ms=3, color=c, alpha=0.6)
    a.set(title=f"bell {name} folded at {Pc:.4f} s", xlabel="time (s)",
          ylim=(0, 1))
    a.grid(alpha=0.2)
ax[0].set_ylabel("phase")
plt.tight_layout()

## 5. The pattern and its variation (circular statistics)

Phase clusters give the strike positions within the cycle; a rigid cycle grid
gives per-position timing residuals. Circular statistics quantify the
periodicity (per-position resultant length R and Rayleigh test) and — the
sharpest result — the **inter-bell phase lock**.

In [7]:
streams = {}
for name in "AB":
    ph0, clusters = rhythm.phase_clusters(strikes[name], Pc)
    for ci, (c, m) in enumerate(clusters):
        streams[f"{name}{ci}"] = strikes[name][m]
grid = rhythm.cycle_grid(streams, Pc, T_STOP)
for n, d in grid["streams"].items():
    print(f"{n}: pos {d['pos']:.3f} s  hit {d['hit_rate']*100:.0f}%  "
          f"sd {d['sd_ms']:.0f} ms  slow-wander {d['slow_sd_ms']:.0f} ms  "
          f"cycle-to-cycle {d['fast_sd_ms']:.0f} ms  lag1 {d['lag1']:.2f}")

A0: pos 0.280 s  hit 93%  sd 110 ms  slow-wander 105 ms  cycle-to-cycle 35 ms  lag1 0.85
A1: pos 2.054 s  hit 91%  sd 156 ms  slow-wander 88 ms  cycle-to-cycle 124 ms  lag1 0.28
A2: pos 1.009 s  hit 28%  sd 149 ms  slow-wander 103 ms  cycle-to-cycle 82 ms  lag1 0.19
A3: pos 1.538 s  hit 9%  sd 85 ms  slow-wander 94 ms  cycle-to-cycle 17 ms  lag1 nan
B0: pos 1.631 s  hit 97%  sd 108 ms  slow-wander 104 ms  cycle-to-cycle 36 ms  lag1 0.84
B1: pos 1.560 s  hit 23%  sd 1037 ms  slow-wander 676 ms  cycle-to-cycle 520 ms  lag1 0.79


In [8]:
def circ(tk, Pc):
    z = np.exp(2j * np.pi * tk / Pc)
    R = abs(z.mean())
    return R, np.sqrt(-2 * np.log(R)) / (2 * np.pi) * Pc * 1e3, \
           np.exp(-len(tk) * R**2)
for n, tk in streams.items():
    R, sd, p = circ(tk, Pc)
    print(f"{n}: R = {R:.3f}, circular sd = {sd:.0f} ms, Rayleigh p < {p:.1e}")
# inter-bell lock: each B strike vs the preceding A strike
tA, tB = streams["A0"], streams["B0"]
i = np.clip(np.searchsorted(tA, tB) - 1, 0, None)
d = ((tB - tA[i]) / Pc) % 1.0
z = np.exp(2j * np.pi * d)
R = abs(z.mean()); mu = (np.angle(z.mean()) / (2 * np.pi)) % 1.0
print(f"\nB relative to A: mean offset {mu:.3f} cycle = {mu*Pc:.3f} s, "
      f"R = {R:.3f}, circular sd = "
      f"{np.sqrt(-2*np.log(R))/(2*np.pi)*Pc*1e3:.0f} ms  -> phase-locked")

A0: R = 0.976, circular sd = 112 ms, Rayleigh p < 6.7e-155
A1: R = 0.953, circular sd = 156 ms, Rayleigh p < 4.7e-144
A2: R = 0.955, circular sd = 153 ms, Rayleigh p < 1.0e-44
A3: R = 0.986, circular sd = 85 ms, Rayleigh p < 6.4e-16
B0: R = 0.977, circular sd = 108 ms, Rayleigh p < 1.1e-161
B1: R = 0.965, circular sd = 135 ms, Rayleigh p < 8.0e-140

B relative to A: mean offset 0.514 cycle = 1.626 s, R = 0.998, circular sd = 33 ms  -> phase-locked


## 6. Tonal content and the cross-talk test

Strike-triggered post/pre spectral **rise** is the tonal fingerprint of one
rhythmic position — and the cross-talk test: an apparent stream whose rise
spectrum contains only the *other* bell's partials is leakage, not a strike.
That is exactly what identifies bell B's "second strike" as an artifact.

In [9]:
from scipy.signal import find_peaks
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), sharex=True, sharey=True)
for a, (n, lab) in zip(ax, [("B0", "B primary (real strike)"),
                            ("B1", "B secondary (cross-talk from A)")]):
    if n not in streams:
        continue
    fq, r = rhythm.rise_spectrum(take, streams[n], n_max=100)
    band = (fq > 350) & (fq < 4500)
    a.plot(fq[band], r[band], lw=0.8, color="#2a78d6")
    a.set(title=lab, xlabel="Hz")
    a.grid(alpha=0.2)
ax[0].set_ylabel("rise (dB)")
plt.tight_layout()
# B0 peaks at 1359/2303/3428 (bell B); B1 peaks at 1131/1910/2871 (bell A!)

## 7. Timbre: partial decay

Median decay slope per partial, 0.1–1.0 s after each strike. The hum decays
an order of magnitude slower than the nominal: the 3.17 s pattern rides on a
continuous B♭+D drone that never dies between strikes.

In [10]:
for gi, cols in enumerate(groups):
    name = "AB"[gi]
    print(f"bell {name}:")
    for c in cols[:6]:
        e = P["env"][:, c]
        slopes = []
        for s in streams[f"{name}0"][::4]:
            i0, i1 = int((s + 0.1) / dt), int((s + 1.0) / dt)
            if i1 < len(t):
                seg = 10 * np.log10(e[i0:i1] + 1e-14)
                slopes.append(np.polyfit(np.arange(len(seg)) * dt, seg, 1)[0])
        sl = np.median(slopes)
        print(f"  {pfreq[c]:7.1f} Hz: {sl:7.1f} dB/s   (T60-eq ~ {-60/sl:.1f} s)"
              if sl < -1 else f"  {pfreq[c]:7.1f} Hz: {sl:7.1f} dB/s")

bell A:
   1130.9 Hz:   -15.6 dB/s   (T60-eq ~ 3.9 s)
   1910.2 Hz:   -30.4 dB/s   (T60-eq ~ 2.0 s)
   2343.8 Hz:   -18.1 dB/s   (T60-eq ~ 3.3 s)
   2437.5 Hz:   -20.5 dB/s   (T60-eq ~ 2.9 s)
   2466.8 Hz:   -20.9 dB/s   (T60-eq ~ 2.9 s)
   2871.1 Hz:   -31.9 dB/s   (T60-eq ~ 1.9 s)
bell B:
   1365.2 Hz:   -19.3 dB/s   (T60-eq ~ 3.1 s)
   2302.7 Hz:   -31.1 dB/s   (T60-eq ~ 1.9 s)
   3029.3 Hz:   -17.3 dB/s   (T60-eq ~ 3.5 s)
   3427.7 Hz:   -34.1 dB/s   (T60-eq ~ 1.8 s)


## 8. Where this ran in the toolbox

Everything above is exposed as `ambiscape rhythm <session>`, which writes
`analysis/rhythm_overview.png` and `analysis/rhythm.json` (including
automatic `crosstalk_suspects` flags). The Schaeffer/Schafer taxonomy
figures, ISO 12913-3 indicators, and the speech-privacy gate used before
publishing the recording are separate ambiscape commands — see the
[full report](RHYTHM_REPORT.md) and the
[case study](https://github.com/fourMs/ambiscape/wiki/Case-Study-Haarlem-Damiaatjes)
on the ambiscape wiki.